# Energy Island System Optimisation (PyPSA · HiGHS)

**Author:** Agus Samsudin  
**Domain:** Energy Systems Modelling &nbsp;·&nbsp; Optimisation &nbsp;·&nbsp; Renewable Energy  
**Tools:** Python · PyPSA · HiGHS · linopy · Pandas · NumPy · Matplotlib · Plotly

---

## Overview

This notebook develops a **Linear Programming (LP) capacity-expansion and dispatch**
model to optimise the generation mix and storage capacity of a renewable-based
**energy island** or **isolated microgrid** system.

The model is built using [**PyPSA**](https://pypsa.org/) (Python for Power System
Analysis), a modern open-source power system modelling framework, and solved
with [**HiGHS**](https://highs.dev/) — a high-performance, open-source LP and
MIP solver accessed through PyPSA's [linopy](https://linopy.readthedocs.io/)
optimisation backend.

The model simultaneously determines optimal **investment decisions** (installed
capacity of each technology) and **operational dispatch** (hourly generation
and storage scheduling) to meet system electricity demand at
**minimum total system cost** or **minimum CO₂ emissions**.

A **grid loss factor** is applied to scale the raw demand up to the gross
generation requirement, accounting for distribution and transmission losses
within the island network.

---

## System Technologies

### Variable Renewable Generation

| Technology | Description |
|---|---|
| **Wind** | Hourly generation profile scaled to 1 MW installed capacity |
| **Solar PV** | Hourly generation profile scaled to 1 MW installed capacity |

Variable renewable technologies provide low-carbon electricity but depend
on resource availability and cannot be dispatched on demand.

### Dispatchable Generation

| Technology | Description |
|---|---|
| **Biomass** | Combustion of solid biomass fuels; considered low-carbon |
| **Biogas** | Anaerobic digestion of organic waste; flexible dispatch |
| **Waste-to-Energy (WTE)** | Energy recovery from municipal solid waste |
| **Hydro** | Run-of-river or reservoir hydro generation |
| **Geothermal** | Baseload generation from geothermal resources |
| **Natural Gas turbine** | Fast-response backup; highest cost and emissions |
| **Biodiesel** | Renewable liquid fuel backup; similar dispatch role to natural gas |

Dispatchable technologies ensure **system reliability** when renewable
generation is insufficient to meet demand.

### Energy Storage

| Technology | Typical Duration | Description |
|---|---|---|
| **BESS** | 2–6 hours | Battery Energy Storage System; fast response, flexible siting |
| **PHS** | 6–12 hours | Pumped Hydro Storage; large-scale, requires suitable topography |
| **Hydrogen** | 12–48+ hours | Long-duration storage via electrolysis and fuel cell/turbine |

Storage technologies improve system flexibility by balancing renewable
variability, shifting energy across time, and reducing curtailment.

---

## Objective

The model supports three optimisation objectives:

| Objective | Description |
|---|---|
| **Lowest LCOE** | Minimises total annualised system cost (default) |
| **Lowest CO₂** | Minimises total annual CO₂ emissions via high carbon shadow price |
| **Most Diversified** | Forces all selected technologies to ≥ 0.5 MW; objective remains lowest cost |

Total system cost includes:

- Capital investment cost (annualised via Capital Recovery Factor)
- Fixed operation and maintenance (O&M) cost
- Variable fuel cost (gas and dispatchable fuels)
- Soft penalty for renewable curtailment
- Soft penalty for unnecessary storage cycling

---

## Model Workflow

| Step | Description |
|:---:|---|
| **1** | Import required Python libraries |
| **2** | Load geographic project information |
| **3** | Configure system technologies and model parameters |
| **4** | Upload technology and resource parameter CSV |
| **5** | Upload hourly demand and generation time series |
| **6** | Build and solve the PyPSA optimisation model |
| **7** | Analyse and visualize results |

---

## Installation

```bash
pip install pypsa highspy linopy pandas numpy matplotlib plotly openpyxl ipywidgets
```

> **Note:** `highspy` is the Python binding for the HiGHS solver.
> PyPSA uses HiGHS automatically when `highspy` is installed.
> No separate solver installation or PATH configuration is required.

---

> **Disclaimer:** This model was developed for educational and research purposes.
> While every effort has been made to ensure accuracy, the author makes no guarantees
> regarding completeness or reliability of results. Use at your own risk.


---

## Step 1 — Required Libraries

The following Python packages are required to run this notebook:

| Library | Purpose |
|---|---|
| **Pandas** | Data loading, handling, and time-series processing |
| **NumPy** | Numerical operations and array handling |
| **Matplotlib** | Visualisation of dispatch, capacity, LCOE, and residual load |
| **PyPSA** | Power system modelling framework for network construction and optimisation |
| **linopy** | LP/MIP modelling layer used internally by PyPSA to interface with HiGHS |
| **Plotly** | Interactive Sankey diagram and energy flow visualisation |
| **ipywidgets** | Interactive file upload and configuration interface |
| **io** | Handling in-memory CSV data from widget file uploads |
| **openpyxl** | Excel export of hourly dispatch results |

The model uses the **HiGHS (High-performance open-source LP/MIP Solver)** —
a modern, high-performance solver that substantially outperforms GLPK on
large-scale LP problems, with significantly faster solve times and superior
numerical stability. HiGHS is accessed directly through PyPSA's linopy
backend without any external binary installation.

A global matplotlib style is configured here so that all charts
share a consistent, professional appearance throughout the notebook.


In [ ]:
# ── Core libraries ────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pypsa
import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go
import logging
logging.getLogger("pypsa").setLevel(logging.WARNING)

# ── Project modules ───────────────────────────────────────────────────────────
from src.style import apply_style
from src.constants import *
from src.geographic import GeographicLoader
from src.setup_options import SetupOptions
from src.resources import ResourceAssessment
from src.timeseries import TimeSeriesData
from src.model import IslandEnergyPyPSA
from src.visualization import ResultsVisualization

apply_style()

---

## Model Constants

All hard-coded assumptions are defined here as named constants.
This is the **only cell** that needs to be edited to change model assumptions —
no values are duplicated elsewhere in the notebook.

| Constant | Value | Source / Notes |
|---|---|---|
| `HOURS_PER_YEAR` | 8760 | Full-year hourly resolution |
| `GRID_LOSS_FACTOR` | 0.04 | Distribution loss factor — 4% of delivered energy lost in the island grid |
| `SOC_MIN_FRACTION` | 0.20 | Minimum state-of-charge — prevents deep discharge damage |
| `SOC_INIT_FRACTION` | 0.50 | Initial and final SoC for cyclic boundary condition |
| `CURTAILMENT_PENALTY` | 1 | €/MWh — small penalty to discourage oversized capacity with curtailment |
| `STORAGE_CHARGE_COST` | 5 | €/MWh — proxy cost to prevent unnecessary storage cycling |
| `CARBON_SHADOW_PRICE` | 10 000 | €/tCO₂ — shadow price applied in Lowest CO₂ objective mode |
| `DIVERSIFIED_MIN_MW`  | 2   | MW — minimum installed capacity per technology in Most Diversified mode |

### Notes on the Grid Loss Factor

The **grid loss factor** (`GRID_LOSS_FACTOR`) accounts for resistive and
transformer losses within the island distribution network. It inflates the
raw consumer demand to the gross **generation requirement** that must be met
at the point of injection:

$$P_{\text{gross}}(t) = P_{\text{demand}}(t) \times (1 + \text{GLF})$$

A 4% loss factor is a typical value for small isolated island networks.
Larger islands with longer distribution lines may have factors of 5–8%.
This value can be adjusted to match measured technical loss data for
the specific system under study.

### Notes on Soft Penalties

The **curtailment penalty** (`CURTAILMENT_PENALTY`) discourages the model
from building oversized capacity and simply curtailing the excess.
It is intentionally small (well below any technology LCOE) so it guides
dispatch without distorting investment decisions.

The **storage charge cost** (`STORAGE_CHARGE_COST`) prevents the solver
from cycling storage unnecessarily when there is no benefit.
Both values are expressed in €/MWh and can be adjusted here if needed.

The **carbon shadow price** (`CARBON_SHADOW_PRICE`) is applied in
"Lowest CO₂" objective mode by adding a very high cost weight to each
tonne of CO₂ emitted. This effectively forces the solver to minimise
emissions while still respecting investment feasibility constraints.

The **diversification minimum** (`DIVERSIFIED_MIN_MW`) sets the floor
capacity (1 MW) that each selected technology must install in
"Most Diversified" mode, ensuring no technology is excluded from the
optimal portfolio.


In [ ]:
# Constants are defined in src/constants.py — printed here for reference.
print(f"  HOURS_PER_YEAR      = {HOURS_PER_YEAR}")
print(f"  GRID_LOSS_FACTOR    = {GRID_LOSS_FACTOR}")
print(f"  SOC_MIN_FRACTION    = {SOC_MIN_FRACTION}")
print(f"  SOC_INIT_FRACTION   = {SOC_INIT_FRACTION}")
print(f"  CURTAILMENT_PENALTY = {CURTAILMENT_PENALTY}")
print(f"  STORAGE_CHARGE_COST = {STORAGE_CHARGE_COST}")
print(f"  CARBON_SHADOW_PRICE = {CARBON_SHADOW_PRICE}")
print(f"  DIVERSIFIED_MIN_MW  = {DIVERSIFIED_MIN_MW}")
print(f"  CONGESTION_THRESHOLD= {CONGESTION_THRESHOLD}")

---

## Step 2 — Geographic Information

This section loads geographic information about the modelled energy system
location. The data provides useful **contextual documentation** for the
project and is displayed for reference.

Geographic coordinates are not directly used in the current optimisation
model, but in future versions could be used to automatically retrieve
solar irradiance, wind resource, and hydrology data via APIs.

### PHS Feasibility Check

The `Max_Height_m` field is particularly important for **Pumped Hydro Storage
(PHS)** feasibility. PHS requires a minimum head difference of approximately
**300 m** between upper and lower reservoirs to be technically and economically
viable. The model will issue a warning if the uploaded location does not
meet this threshold.

**Expected file:** `geographic.csv` (upload via widget below)

**Required columns:**

| Column | Type | Description |
|---|---|---|
| `Name` | string | Location name (island, region, or project name) |
| `Latitude` | float | Decimal degrees north |
| `Longitude` | float | Decimal degrees east |
| `Max_Height_m` | float | Maximum terrain elevation difference (metres) — used for PHS feasibility |

**Example:**
```
Name,Latitude,Longitude,Max_Height_m
El Hierro,27.74,-18.03,1500
```


In [ ]:
geo = GeographicLoader()
geo.upload()

---

## Step 3 — System Configuration

This section provides an interactive widget to configure the energy system
for optimisation. Select the technologies to include, specify storage
duration limits, set the optimisation objective, and confirm the discount rate.

### Technology Selection

Technologies are grouped into three categories:

| Category | Technologies | Notes |
|---|---|---|
| **Generation Technologies** | Wind, Solar, Biomass, Biogas, Geothermal, Hydro, WTE | Each uses a normalised hourly profile; all require a matching time-series file |
| **Energy Storage** | BESS, PHS, Hydrogen | User sets maximum discharge duration (hours); power and energy capacity are co-optimised |
| **Balancing Technology** | Natural Gas, Biodiesel | Peaking units providing firm capacity backup at high marginal cost |

### Storage Duration Parameter

The **max duration hours** parameter sets the upper bound on the
energy-to-power ratio (E/P ratio) for each storage technology.
The optimiser may select a shorter duration if it minimises cost.

In the PyPSA model, the combined capital cost per MW of storage power
capacity is pre-computed as:

$$\text{capital\_cost} = (K^{\text{MW}} \cdot \text{CRF} + O\&M)
+ K^{\text{MWh}} \cdot \text{CRF} \cdot \text{MaxHours}$$

where the energy cost component is scaled by the user-specified maximum
duration. This is a standard formulation for storage optimisation when
the E/P ratio is fixed or bounded.


### Demand Scaling

The **demand scale (%)** parameter multiplies every hourly value in the
uploaded demand CSV by a constant factor before it is passed to the model.
This allows scenario analysis — growth, reduction, or sensitivity testing —
without editing the underlying data file.

| Value | Effect |
|---|---|
| `100%` | Baseline demand — CSV loaded as-is (default) |
| `110%` | +10% demand growth scenario |
| `90%` | −10% demand reduction / efficiency scenario |
| `150%` | +50% electrification / EV uptake scenario |

The scaling is applied **after** loading and validation, so the raw CSV is
always checked at its original values. The scaled peak demand and annual
total are printed when the file is loaded, confirming the adjustment.

$$P_{\text{scaled}}(t) = P_{\text{raw}}(t) \times \frac{\text{DemandScale}}{100}$$

### Discount Rate

The **discount rate** is used in the Capital Recovery Factor (CRF) to
annualise investment costs over each technology's economic lifetime.
A rate of 8% (0.08) is a commonly used default for energy planning
in developing regions and island systems.


In [ ]:
setup = SetupOptions()
setup.display()

---

## Step 4 — Technology and Resource Parameters

This section loads the **techno-economic characteristics** of each technology
included in the optimisation. These parameters define the cost structure,
physical limits, and environmental performance of each technology.

### Required File

Upload a CSV file with **one row per technology**. The `Sources` column must
exactly match the technology names selected in Step 3.

### Required Columns

| Column | Unit | Description |
|---|---|---|
| `Sources` | — | Technology name (must match Step 3 selections exactly) |
| `Max_Capacity_MW` | MW | Maximum installable capacity (resource or land constraint) |
| `Investment_per_MW` | €/MW | Capital cost per MW of installed power capacity |
| `O&M_per_MW_yr` | €/MW/yr | Annual fixed operation and maintenance cost |
| `Lifetime` | years | Economic asset lifetime (used in CRF calculation) |
| `CO2_per_MWh` | tCO₂/MWh | Direct CO₂ emissions per MWh of electricity generated |
| `Fuel_Cost` | €/MWhₜℏᵤᵉˡ | Variable fuel cost per MWh of fuel consumed (0 for renewables) |
| `Efficiency` | 0–1 | Generator thermal efficiency or storage one-way efficiency |
| `Merit_Order` | integer | Dispatch priority (lower = dispatched first) |
| `Storage_MWh` | €/MWh | Capital cost per MWh of energy storage capacity (storage only; 0 for generators) |

### Notes

- For **renewable generators**, `Fuel_Cost` and `CO2_per_MWh` should be set to 0
- For **storage technologies**, both `Investment_per_MW` (power capacity cost)
  and `Storage_MWh` (energy capacity cost) must be provided
- `Efficiency` for storage represents the **one-way efficiency**;
  round-trip efficiency = efficiency²
- Technology parameter data can be sourced from
  [IRENA Cost Database](https://www.irena.org/costs),
  [NREL ATB](https://atb.nrel.gov/), or peer-reviewed literature


In [ ]:
resources = ResourceAssessment()
resources.upload()

---

## Step 5 — Time Series Data

The optimisation model requires **hourly time series data** describing
electricity demand and the availability of each renewable resource.
All datasets must contain exactly **8760 hourly values** representing
one complete year.

### Required Datasets

| Dataset | Unit | Description |
|---|---|---|
| **Electricity demand** | MW | Hourly system demand (not normalised) — scaled by the Demand Scale % set in Step 3 |
| **Generation profile per technology** | MW / MWᴵⁿˢᵗˡˡᵉᵈ | Normalized output per 1 MW installed — one file per selected technology |

### Generation Profile Format

Generation profiles represent the **normalised hourly output** of each
technology per 1 MW of installed capacity (values between 0 and 1).
The profile already represents **electrical output** — generator efficiency
is embedded in the profile values and must **not** be applied again.

In PyPSA, these profiles are passed directly as the `p_max_pu` time series
(availability factor) for each generator:

$$p_{\text{max\_pu}}(t) = \text{profile}(t)$$

The PyPSA optimiser then determines the actual dispatch:

$$P_{\text{gen}}(t) = p_{\text{nom\_opt}} \times p_{\text{max\_pu}}(t) \geq 0$$

Generator efficiency ($\eta$) is used **only** in the economic calculation
to convert fuel cost from €/MWh$_{\text{fuel}}$ to €/MWh$_{\text{electric}}$:

$$c_{\text{mc}} = \frac{\text{Fuel\_Cost}}{\eta}$$

Any unused available generation is implicitly curtailed by the solver.

### Data Sources

Hourly renewable generation profiles and climate data can be obtained from:

| Source | Data Available |
|---|---|
| [Renewables.ninja](https://www.renewables.ninja) | Wind and solar PV generation profiles |
| [Global Solar Atlas](https://globalsolaratlas.info) | Solar irradiance and PV yield data |
| [PVGIS](https://re.jrc.ec.europa.eu/pvg_tools/) | European Commission PV estimation tool |
| [ERA5 / Copernicus](https://cds.climate.copernicus.eu) | Global climate reanalysis data |
| [IRENA Resource Map](https://resourceirena.irena.org) | Renewable resource atlases |

### Demand Scaling

The demand CSV is loaded at its original values and validated first.
The **Demand Scale %** configured in Step 3 is then applied to produce
the scaled demand array used by the model. The loader prints the raw
and scaled peak and annual totals for confirmation.

### Validation

Each uploaded file is automatically checked for:
- Correct row count (exactly 8760 rows)
- Absence of NaN or missing values
- Non-negative values throughout


In [ ]:
ts = TimeSeriesData(setup)

print("Generation Profiles")
print("Edit paths below to match your data/time_series/ folder, then click Load.")
ts.upload_generation()

print("\nDemand Profile")
ts.upload_demand()

---

## Step 6 — Optimisation Model (PyPSA · HiGHS)

The system is optimised using a **Linear Programming (LP) capacity-expansion
and dispatch** formulation built with [PyPSA](https://pypsa.org/) and solved
with the high-performance open-source **HiGHS** solver via the
[linopy](https://linopy.readthedocs.io/) backend.

### PyPSA Network Architecture

The island system is represented as a **single-bus AC network** — a common
and well-established approximation for isolated microgrids where transmission
constraints within the island are not the primary focus of study.

| Component | PyPSA Class | Description |
|---|---|---|
| Island grid | `Bus` | Single AC bus representing the entire island system |
| Gross demand | `Load` | Hourly demand scaled up by the grid loss factor |
| VRE generators | `Generator` | Wind, solar — `p_nom_extendable`, time-varying `p_max_pu` |
| Non-flexible generators | `Generator` | Biomass, Biogas, Geothermal, WTE — `p_nom_extendable`; `p_min_pu = p_max_pu` (must follow profile exactly) |
| Flexible dispatchable | `Generator` | Hydro — `p_nom_extendable`, free dispatch 0 → `p_max_pu` |
| Natural Gas backup | `Generator` | Peaking unit — `p_nom_extendable`, high marginal cost |
| Biodiesel backup | `Generator` | Renewable fuel peaker — `p_nom_extendable`, high marginal cost |
| Storage | `StorageUnit` | BESS, PHS, Hydrogen — `p_nom_extendable`, cyclic SoC |

### Decision Variables

**Investment variables** (capacity sizing):

| Variable | Unit | PyPSA Attribute | Description |
|---|---|---|---|
| `p_nom_opt[g]` | MW | `n.generators.p_nom_opt` | Optimal installed generation capacity |
| `p_nom_opt[s]` | MW | `n.storage_units.p_nom_opt` | Optimal storage power capacity |
| energy capacity | MWh | `p_nom_opt × max_hours` | Derived optimal energy capacity |

**Operational variables** (dispatch scheduling):

| Variable | Unit | PyPSA Attribute | Description |
|---|---|---|---|
| `p[g, t]` | MW | `n.generators_t.p` | Hourly generation dispatch |
| `p[s, t]` | MW | `n.storage_units_t.p` | Net storage power (+ discharge, – charge) |
| `soc[s, t]` | MWh | `n.storage_units_t.state_of_charge` | Storage state of charge |

### Objective Function

**Lowest LCOE mode** — minimise total annualised system cost:

$$\min \sum_{g} p_{\text{nom}}^{g} \cdot c_{\text{cap}}^{g}
+ \sum_{s} p_{\text{nom}}^{s} \cdot c_{\text{cap}}^{s}
+ \sum_{g,t} p_{g,t} \cdot c_{\text{mc}}^{g}
+ \sum_{s,t} p_{s,t}^{+} \cdot c_{\text{stor}}$$

where $c_{\text{cap}}$ is the annualised capital cost (CRF × CAPEX + O&M),
and the **Capital Recovery Factor** annualises investment over asset lifetime:

$$\text{CRF} = \frac{r(1+r)^n}{(1+r)^n - 1}$$

**Lowest CO₂ mode** — adds a large carbon shadow price to marginal costs,
heavily penalising CO₂-intensive technologies:


$$c_{\text{mc}}^{g} \leftarrow c_{\text{mc}}^{g} + \text{CO}_{2,g} \times \text{CSP}$$

where CSP = 10 000 €/tCO₂ is the carbon shadow price.

**Most Diversified mode** — keeps the standard lowest-cost objective but enforces a
mandatory minimum installed capacity of **1 MW** on every technology selected in
Step 3, implemented via `p_nom_min = DIVERSIFIED_MIN_MW` on each PyPSA Generator and StorageUnit.
The solver remains free to invest beyond 1 MW wherever economically justified,
but cannot exclude any checked technology from the optimal portfolio.

### Key Constraints

| Constraint | Description |
|---|---|
| **Power balance** | Sum of generation + storage dispatch = gross demand (every hour) |
| **Capacity limits** | Generator dispatch ≤ `p_max_pu(t) × p_nom_opt` |
| **Max capacity bounds** | `p_nom_opt ≤ Max_Capacity_MW` (resource constraint) |
| **Storage power limits** | Charge/discharge ≤ `p_nom_opt` |
| **SoC dynamics** | $\text{SoC}_{s,t} = \text{SoC}_{s,t-1} + \eta \cdot \text{charge}_{s,t} - \text{discharge}_{s,t} / \eta$ |
| **SoC minimum** | $\text{SoC}_{s,t} \geq \text{SOC\_MIN} \times p_{\text{nom}} \times \text{MaxHours}$ |
| **SoC maximum** | $\text{SoC}_{s,t} \leq p_{\text{nom}} \times \text{MaxHours}$ |
| **Cyclic SoC** | Initial SoC = final SoC (enforced via `cyclic_state_of_charge=True`) |

### Grid Loss Factor Integration

The gross demand passed to the PyPSA `Load` component is:

$$P_{\text{load}}(t) = P_{\text{demand}}(t) \times (1 + \text{GLF})$$

This ensures that all installed generation capacity is sized to cover
both consumer demand and distribution losses simultaneously, which
increases the optimal generation capacity and system cost by approximately
GLF = 4% relative to a lossless model.

### Note on Storage Capital Cost Formulation

PyPSA's `StorageUnit` component parameterises storage with a fixed
energy-to-power ratio (`max_hours`). The per-MW capital cost is
pre-computed to aggregate both power and energy cost components:

$$c_{\text{cap}}^{s} = K_{\text{MW}}^{s} \cdot \text{CRF}_s + O\&M_s
+ K_{\text{MWh}}^{s} \cdot \text{CRF}_s \cdot \text{MaxHours}_s$$

The optimiser determines the optimal power capacity `p_nom_opt`;
the energy capacity is derived as `p_nom_opt × max_hours`.
This formulation is equivalent to the original two-variable formulation
when the maximum duration constraint is binding.


In [ ]:
model = IslandEnergyPyPSA(setup, resources, ts)
model.build()
model.solve()

---

## Step 7 — File Export

After reviewing the visualisation results, the optimised model outputs can be
exported to two file formats for archiving, reporting, and downstream use.

### Excel Export

The `export_results()` method writes a single-sheet Excel workbook with
**8 760 rows** (one per hour) and one column per output timeseries:

| Column pattern | Unit | Description |
|---|---|---|
| `Prod_{tech}_MW` | MW | Hourly generation dispatch per technology |
| `Discharge_{storage}_MW` | MW | Storage discharge (positive flow to grid) |
| `Charge_{storage}_MW` | MW | Storage charge (positive flow from grid) |
| `SOC_{storage}_MWh` | MWh | State of charge at end of each hour |
| `GrossDemand_MW` | MW | Demand including grid losses fed to the model |
| `NetDemand_MW` | MW | Raw consumer demand (before loss factor) |
```python
model.export_results("results/optimisation_results.xlsx")
```

### Dashboard JSON Export

The `export_dashboard_json()` method writes a structured JSON file optimised for
**dashboard and downstream application consumption**. The file contains
six top-level sections:

| Key | Contents |
|---|---|
| `meta` | Objective, discount rate, currency, grid loss factor, technology lists |
| `capacities` | Installed MW (and MWh for storage) per technology |
| `energy_mix` | Annual generation, capacity factor, per-technology LCOE, CO₂ |
| `lcoe_summary` | Per-technology LCOE/LCOS and system LCOE |
| `co2_summary` | Per-technology and system CO₂ totals and emission intensity |
| `hourly` | Full 8 760-row array for all generators, storage, and demand |
```python
model.export_dashboard_json("results/dashboard_results.json", geo=geo)
```

> **Tip:** Edit the filepath arguments to organise results by scenario,
> e.g. `"results/scenario_diversified.xlsx"` and
> `"results/scenario_diversified.json"`.


In [ ]:
model.export_results("results/optimisation_results.xlsx")
model.export_dashboard_json("results/dashboard_results.json", geo=geo)

---

## Step 8 — Results Visualisation

After solving the optimisation problem, results are summarised and visualised
through a comprehensive set of text summaries and charts. All result data
is extracted directly from the solved `pypsa.Network` object.

### Export

| Method | Output | Description |
|--------|--------|-------------|
| `export_results(path)` | `.xlsx` | Hourly dispatch, storage SoC, and demand |
| `export_dashboard_json(path)` | `.json` | Structured JSON for dashboard: meta, capacities, energy mix, LCOE, CO₂, hourly data |

### Text Summaries

| Method | Description |
|--------|-------------|
| `summary()` | Installed capacity of all generation and storage technologies |
| `calculate_lcoe()` | Per-technology LCOE/LCOS and overall system LCOE |

### Charts

| Method | Chart Type | Description |
|--------|-----------|-------------|
| `plot_installed_capacity()` | Bar chart | Installed MW per generation technology |
| `plot_storage_capacity()` | Side-by-side bars | Storage power (MW) and energy (MWh) capacity |
| `plot_energy_mix()` | Donut chart | Annual generation mix as percentage of total demand |
| `plot_capacity_factors()` | Horizontal bars | Achieved capacity factor per technology |
| `plot_dispatch(hours)` | Stacked area | Hourly dispatch by technology with demand line |
| `plot_soc(hours)` | Line chart | Storage state of charge with capacity reference bands |
| `plot_residual(hours)` | Area chart | Residual load after renewables (deficit/surplus) |
| `plot_load_duration()` | Step chart | Sorted load duration curve with generation breakdown |
| `plot_worst_residual_week()` | Combined | Auto-detects and plots the hardest week for renewables |
| `plot_energy_sankey()` | Sankey | Interactive annual energy flow diagram |

### Dispatch Chart: Stack Order

The `plot_dispatch()` chart stacks technologies in merit order (lowest cost first):

| Layer (bottom → top) | Color | Description |
|---|---|---|
| WTE, Geothermal, Hydro | Dark tones | Baseload dispatchable renewables |
| Biomass, Biogas | Green tones | Flexible dispatchable renewables |
| Wind, Solar | Blue / amber | Variable renewable generation |
| PHS, BESS, Hydrogen | Cyan / green | Storage discharge |
| Natural Gas | Grey | Backup balancing generation |
| Biodiesel | Warm amber | Renewable fuel backup generation |

### Worst Renewable Week

The `plot_worst_residual_week()` method automatically identifies the
168-hour window with the **highest cumulative residual load** — i.e. the
week where demand is least covered by renewable generation. This is the
most critical period for storage adequacy and gas backup sizing.

### How to Use

Pass `hours` as a `(start, end)` tuple or a single integer for a
168-hour window starting at that hour:

```python
viz.plot_dispatch((1000, 1168))   # explicit window
viz.plot_dispatch(1000)           # 168-hour window from hour 1000
```
| `plot_grid_congestion()` | Multi-panel | Bus utilisation ratio, duration curve, monthly congested hours |

### Grid Congestion Analysis

The `plot_grid_congestion()` method analyses system stress by computing the
hourly **bus utilisation ratio** — the ratio of gross demand to total
available supply capacity at the island bus:

$$\text{utilisation}(t) = \frac{P_{\text{gross}}(t)}{\sum_g p_{\text{nom}}^g \cdot p_{\text{max\_pu}}^g(t) + \sum_s p_{\text{nom}}^s}$$

Hours where utilisation exceeds the configurable threshold (default 80%)
are flagged as "congested" — the system has thin headroom and is vulnerable
to forecast errors, unplanned outages, or demand spikes.

The analysis produces three panels:

| Panel | Description |
|---|---|
| **Timeseries** | Hourly utilisation with congested hours shaded red |
| **Duration curve** | Utilisation sorted from highest to lowest |
| **Monthly bar** | Number of congested hours per calendar month |

The congestion data is also exported to the dashboard JSON under the
`grid_congestion` key for downstream visualisation.


In [ ]:
# ── Run all results ────────────────────────────────────────────────────────────
viz = ResultsVisualization(model, setup, resources, ts)

viz.summary()
viz.calculate_lcoe()
viz.plot_energy_mix()
viz.plot_installed_capacity()
viz.plot_storage_capacity()
viz.plot_capacity_factors()
viz.plot_lcoe_breakdown()
viz.plot_dispatch((4000, 4300))         # adjust window as needed
viz.plot_load_duration()
viz.plot_demand_heatmap()
viz.plot_monthly_cf()
viz.plot_energy_sankey()
viz.plot_soc(1000)
viz.plot_residual(1000)

viz.plot_worst_residual_week()

viz.plot_grid_congestion()

---

## Model Summary

This notebook demonstrates a **techno-economic optimisation** of a renewable-based
island energy system using Linear Programming (LP) capacity expansion and dispatch,
implemented in the **PyPSA** power system modelling framework and solved with
the **HiGHS** open-source solver.

The model determines the **least-cost system configuration** of:

- **Generation capacity** — optimal installed MW for each technology
- **Storage capacity** — optimal power (MW) and energy (MWh) sizing
- **Operational dispatch** — hourly scheduling of generation and storage
- **Grid losses** — distribution loss factor applied to gross demand

while satisfying hourly electricity demand and all technical constraints,
and minimising total annualised system cost or CO₂ emissions.

---

### Potential Applications

| Application | Description |
|---|---|
| Renewable energy planning | Optimal technology mix for island or remote systems |
| Microgrid design | Off-grid or grid-connected community energy systems |
| Island power systems | Replacement of diesel with renewables |
| Storage evaluation | Role and sizing of BESS, PHS, and hydrogen |
| Renewable integration | Analysis of high-VRE systems with storage |
| Academic research | Teaching and research in energy systems modelling |

---

### Potential Extensions

The model is designed to be **modular and extensible**. Future versions could include:

| Extension | Description |
|---|---|
| Multi-bus network | Line-constrained dispatch for multi-zone island systems |
| Stochastic optimisation | Uncertainty in demand, wind, and solar resources |
| Multi-year investment | Phased capacity expansion over a planning horizon |
| CO₂ budget constraint | Hard emission cap via `n.add("GlobalConstraint", ...)` |
| Demand response | Flexible loads as a system balancing resource |
| EV fleet integration | Electric vehicles as distributed storage |
| Line loss modelling | Explicit resistive losses on internal distribution lines |

---

### References

- Brown, T. et al. (2018). PyPSA: Python for Power System Analysis. *Journal of Open Research Software*, 6(1), 4.
- Huangfu, Q. & Hall, J.A.J. (2018). Parallelizing the dual revised simplex method. *Mathematical Programming Computation*, 10, 119–142. [HiGHS]
- IRENA (2023). *Renewable Power Generation Costs*. International Renewable Energy Agency.
- IEA (2023). *World Energy Outlook*. International Energy Agency.
- NREL (2023). *Annual Technology Baseline*. National Renewable Energy Laboratory.
- Pfenninger, S. & Staffell, I. (2016). Long-term patterns of European PV output.
  *Energy*, 114, 1251–1265. [Renewables.ninja]

---

*Developed by Agus Samsudin — Energy Systems Modelling*
